In [98]:
pip install langgraph langchain langchain-google-genai


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [99]:
pip install langchain-groq


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [100]:
class WorldGraphNode:
    
    def __init__(self, domain: str, attributes: dict):
        self.domain = domain
        self.attributes = attributes

    def update(self, attribute_name: str, state: str, value):
        if attribute_name not in self.attributes:
            self.attributes[attribute_name] = {}
        self.attributes[attribute_name][state] = value

    def show(self):
        print(f"Domain: {self.domain}")
        print("Attributes:")
        for attr, states in self.attributes.items():
            print(f"  - {attr}:")
            if isinstance(states, dict):
                for state, val in states.items():
                    print(f"      {state}: {val}")
            else:
                print(f"      {states}")
                
    def to_text(self):
        # Start with the domain
        text = f"Domain: {self.domain}\n"
        
        # Loop over every attribute
        for attr, states in self.attributes.items():
            text += f"- {attr}:\n"
            
            # Ensure it is a dictionary before iterating over states
            if isinstance(states, dict):
                for state, val in states.items():
                    # 4 spaces of indentation for the state
                    text += f"    {state}: {val}\n"
            else:
                # Fallback if state is just a raw string/value
                text += f"    {states}\n"
                
        # Return the generated text, stripping any trailing newline at the very end
        return text.strip()

    
    


In [101]:
# Create a medical node
node = WorldGraphNode(
    domain="medical",
    attributes={
        "allergy": {
            "past": "penicillin",
            "current": None,
            "future": None
        }
    }
)

# Show initial state
node.show()

# User got treatment - update current state
node.update("allergy", "current", "cleared")

print("\nAfter treatment:")
node.show()

# Add a new attribute entirely
node.update("blood_type", "current", "B+")

print("\nAfter adding blood type:")
node.show()

Domain: medical
Attributes:
  - allergy:
      past: penicillin
      current: None
      future: None

After treatment:
Domain: medical
Attributes:
  - allergy:
      past: penicillin
      current: cleared
      future: None

After adding blood type:
Domain: medical
Attributes:
  - allergy:
      past: penicillin
      current: cleared
      future: None
  - blood_type:
      current: B+


In [102]:
node = WorldGraphNode(
    domain="medical",
    attributes={
        "allergy": {
            "past": "penicillin",
            "current": "cleared",
            "future": None
        },
        "blood_type": {
            "current": "B+"
        }
    }
)

print(node.to_text())

Domain: medical
- allergy:
    past: penicillin
    current: cleared
    future: None
- blood_type:
    current: B+


In [104]:
from sentence_transformers import SentenceTransformer
import numpy as np

class WorldGraph:
    def __init__(self):
        self.nodes = {}
        self.embeddings = {}
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
    
    def add_node(self, node: WorldGraphNode):
        self.nodes[node.domain] = node
        text = node.to_text()
        self.embeddings[node.domain] = self.model.encode(text)
    
    def retrieve(self, query: str, top_k: int = 2, threshold: float = 0.3):
        # 1. Encode query
        query_embedding = self.model.encode(query).reshape(1, -1)
        scores = []
        
        # 2. Compute similarity for all nodes
        for domain, node_embedding in self.embeddings.items():
            node_emb_2d = node_embedding.reshape(1, -1)
            similarity = np.dot(query_embedding, node_emb_2d.T) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(node_emb_2d)
            )
            scores.append((domain, similarity[0][0]))
            
        # 3. FIX: Filter out nodes that don't meet the minimum similarity threshold
        # This stops the LLM from receiving totally irrelevant context
        scores = [(domain, score) for domain, score in scores if score >= threshold]
        
        # 4. Sort remaining scores
        scores.sort(key=lambda x: x[1], reverse=True)
        
        # 5. Return only the valid top_k matches
        return [self.nodes[d] for d, s in scores[:top_k]]


    
    def update_node(self, domain: str, attribute_name: str, state: str, value):
        # 1. Check if the node exists
        node = self.nodes.get(domain)
        
        # 2. If it doesn't exist, create it!
        if not node:
            node = WorldGraphNode(domain=domain, attributes={})
            self.nodes[domain] = node
            
        # 3. Proceed with the update and embedding refresh normally
        node.update(attribute_name, state, value)
        self.embeddings[domain] = self.model.encode(node.to_text())


In [105]:
# Build the graph
graph = WorldGraph()

# Add nodes
medical_node = WorldGraphNode(
    domain="medical",
    attributes={
        "allergy": {
            "past": "penicillin",
            "current": "cleared",
            "future": None
        }
    }
)

location_node = WorldGraphNode(
    domain="location",
    attributes={
        "city": {
            "past": "Mumbai",
            "current": "Delhi",
            "future": None
        }
    }
)

financial_node = WorldGraphNode(
    domain="financial",
    attributes={
        "income": {
            "current": "45000/year"
        }
    }
)

graph.add_node(medical_node)
graph.add_node(location_node)
graph.add_node(financial_node)

# Retrieve relevant nodes for a query
query = "I've been feeling sick lately"
results = graph.retrieve(query, top_k=2)

print(f"Query: {query}")
print(f"Retrieved nodes:")
for node in results:
    print(node.to_text())
    print("---")

Query: I've been feeling sick lately
Retrieved nodes:


In [106]:
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, List
import json
import os

# Your existing classes
# WorldGraphNode and WorldGraph go here

# Initialize Gemini
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=""
)

# Initialize WorldGraph
graph = WorldGraph()

In [ ]:
SYSTEM_PROMPT = """Role & Objective: You are a helpful assistant with access to a long-term memory system called the WorldGraph. Your goal is to converse with the user while maintaining an accurate, up-to-date record of their world state.

1. Your Input:
Every turn, you will receive two pieces of information:
USER_MESSAGE: The text the user just sent.
WORLD_GRAPH_CONTEXT: A text-based representation of relevant memories retrieved from your graph (organized by Domain, Attribute, and State: Past/Current/Future).

2. How to Process Context:
- Use the Context to Answer: When a user asks a question, look first at the WORLD_GRAPH_CONTEXT. If the answer is there, use it to personalize your response. (e.g., If context shows allergy: {current: penicillin}, and they ask "Can I take this med?", check for penicillin).
- Identify State Changes: If the user provides new facts (e.g., "I moved," "I'm planning to travel," "I used to smoke"), check if this contradicts or updates the WORLD_GRAPH_CONTEXT.

3. Output Requirements:
Your response must contain two parts in this exact order:

PART A: Natural Language Response. Talk to the user normally. Acknowledge their statement or answer their question based on the retrieved context. DO NOT label this section. DO NOT write "PART A" or "Response". Just write your reply.

PART B: Graph Update (JSON Block). If—and only if—the user's message contains new factual information that changes their world state, you MUST append a JSON block at the very end of your message. 

The JSON must follow this exact schema:
```json
{
  "domain": "The high-level category (e.g., 'medical', 'location', 'career')",
  "attribute_name": "The specific item being tracked (e.g., 'residence', 'allergy', 'job_title')",
  "state": "Must be exactly 'past', 'current', or 'future'",
  "value": "The new specific value (e.g., 'Bangalore', 'None', 'Software Engineer')"
}
CRITICAL RULES:

1.If no new information is provided, do NOT include a JSON block.
2.NEVER include labels like 'PART A:', 'PART B:', or 'Graph Update:' anywhere in your output.
3.If the WORLD_GRAPH_CONTEXT does not contain information relevant to the user's message, do not invent or assume facts. Say you don't have that information yet."""


In [ ]:
from typing import TypedDict, List

class AgentState(TypedDict):
    user_message: str
    retrieved_context: list
    response: str
    graph_update: dict

In [ ]:
def input_node(state: AgentState) -> AgentState:
    return state

In [ ]:
def retrieve_node(state: AgentState) -> AgentState:
    # 1. Get the message from the state
    message = state["user_message"]
    
    # 2. Search the WorldGraph using the message
    result = graph.retrieve(message)
    
    # 3. Store the retrieved nodes back into the state
    state["retrieved_context"] = result
    
    # 4. Return the updated state
    return state


In [ ]:
import json

def generate_node(state: AgentState) -> AgentState:
    user_message = state["user_message"]
    context_text = "\n---\n".join([
        node.to_text() for node in state["retrieved_context"]
    ])
    
    prompt = f"""{SYSTEM_PROMPT}

WORLD_GRAPH_CONTEXT:
{context_text}

USER_MESSAGE:
{user_message}
"""
    response = llm.invoke(prompt)
    response_text = response.content
    
    json_start = response_text.find("{")
    
    if json_start != -1:
        # 1. Clean up Part A completely
        Part_A = response_text[:json_start].strip()
        Part_A = Part_A.replace("PART A: Natural Language Response", "") \
                       .replace("PART A:", "") \
                       .replace("PART B: Graph Update", "") \
                       .strip()
        
        # 2. Extract and parse Part B
        Part_B_raw = response_text[json_start:]
        Part_B = Part_B_raw.replace("```json", "").replace("```", "").strip()
        
        try:
            updated_data = json.loads(Part_B)
            state["response"] = Part_A
            state["graph_update"] = updated_data
        except Exception as e:
            state["response"] = Part_A
            state["graph_update"] = None
    else:
        # No JSON found, remove structural labels from the whole text
        Part_A = response_text.replace("PART A: Natural Language Response", "") \
                              .replace("PART A:", "") \
                              .replace("PART B: Graph Update", "") \
                              .strip()
        state["response"] = Part_A
        state["graph_update"] = None
        
    return state


In [ ]:
def update_node(state: AgentState) -> AgentState:
    print(f"graph_update received: {state['graph_update']}")
    if state.get("graph_update") is not None:
        update_data = state["graph_update"]
        graph.update_node(
            domain=update_data["domain"],
            attribute_name=update_data["attribute_name"],
            state=update_data["state"],
            value=update_data["value"]
        )
    return state

In [ ]:
from langgraph.graph import StateGraph, END

# Build the graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("input", input_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)
workflow.add_node("update", update_node)

# Add edges
workflow.add_edge("input", "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "update")
workflow.add_edge("update", END)

# Set entry point
workflow.set_entry_point("input")

# Compile
app = workflow.compile()

print("Agent compiled successfully")

Agent compiled successfully


In [ ]:
def chat(user_message: str):
    result = app.invoke({
        "user_message": user_message,
        "retrieved_context": [],
        "response": "",
        "graph_update": None
    })
    return result["response"]

In [ ]:
medical_node = WorldGraphNode(
    domain="medical",
    attributes={
        "allergy": {
            "past": "penicillin",
            "current": None,
            "future": None
        }
    }
)
graph.add_node(medical_node)
print("WorldGraph initialized with medical node")

WorldGraph initialized with medical node


In [ ]:
print("Agent ready. Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        break
    response = chat(user_input)
    print(f"Agent: {response}\n")

Agent ready. Type 'quit' to exit.

graph_update received: {'domain': 'location', 'attribute_name': 'residence', 'state': 'current', 'value': 'Delhi'}
Agent: You're adjusting your location. I've taken note of your move from Mumbai to Delhi. Since you just moved, I don't have any information on your current residence yet, but I'll keep this in mind for our future conversations.



In [ ]:
print(graph.nodes.keys())


dict_keys(['medical'])


In [ ]:
print(graph.nodes.keys())
graph.nodes["location"].show()

dict_keys(['medical'])


KeyError: 'location'

In [ ]:
results = graph.retrieve("I just moved to Bangalore from Delhi")
for node in results:
    print(node.to_text())
    print("---")

Domain: medical
- allergy:
    past: penicillin
    current: None
    future: None
---
